# Streamlit for Chatbot — Materi Training

Source repo: [adiptamartulandi/chatbot-streamlit-demo](https://github.com/adiptamartulandi/chatbot-streamlit-demo)

## Apa itu Streamlit?

**Streamlit** adalah framework Python open-source yang memungkinkan kamu membangun web app interaktif hanya dengan Python — tanpa perlu HTML, CSS, atau JavaScript sama sekali.

Cukup tulis skrip Python biasa, dan Streamlit akan otomatis mengubahnya menjadi tampilan web yang interaktif.

Ini sangat cocok untuk:
- Demo model machine learning
- Dashboard data
- Aplikasi chatbot berbasis LLM
- Prototipe cepat yang bisa langsung dibagikan ke orang lain

## Kenapa Streamlit untuk Chatbot?

| Pendekatan | Kompleksitas | Kecepatan build |
|---|---|---|
| Flask + HTML/JS | Tinggi | Lama |
| FastAPI + React | Sangat tinggi | Sangat lama |
| **Streamlit** | Rendah | Cepat (menit) |

Streamlit punya `st.chat_message()` dan `st.chat_input()` yang dirancang khusus untuk membangun antarmuka chatbot dengan sedikit baris code.

## Struktur Notebook Ini

1. **Setup** — install Streamlit dan konfigurasi ngrok untuk menjalankan di Google Colab
2. **Part 1: Komponen Dasar Streamlit** — pengenalan elemen-elemen UI yang tersedia
3. **Part 2: Chatbot dengan Gemini** — membangun chatbot lengkap dengan session state dan Gemini API

## Kenapa Pakai ngrok?

Google Colab tidak punya IP publik. Streamlit berjalan di port 8501 di dalam server Colab,
tapi kita tidak bisa mengaksesnya langsung dari browser.

**ngrok** membuat "tunnel" dari internet ke port tersebut — sehingga app kamu bisa diakses
via URL publik tanpa install apapun di komputer lokal.

```
Browser kamu
    ↕  (HTTPS)
  ngrok server  ←→  tunnel  ←→  Google Colab (port 8501)
                                       ↕
                               Streamlit app berjalan
```

## Setup: Install Library & Konfigurasi ngrok

### Step 1 — Install Streamlit dan pyngrok

In [ ]:
!pip install -q streamlit pyngrok google-genai

### Step 2 — Daftarkan ngrok Auth Token

Cara mendapatkan token:
1. Daftar gratis di [https://ngrok.com](https://ngrok.com)
2. Login → klik **Your Authtoken** di dashboard
3. Copy token-nya, paste ke Colab Secrets dengan nama `NGROK_TOKEN`

> Token ngrok gratis sudah cukup untuk keperluan training.
> Satu akun ngrok bisa membuka 1 tunnel aktif sekaligus.

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
print("ngrok token berhasil dikonfigurasi!")

### Fungsi Helper: Jalankan Streamlit + Buka Tunnel

Fungsi ini melakukan tiga hal:
1. Menulis file `.py` ke disk
2. Menjalankan `streamlit run` di background
3. Membuka tunnel ngrok dan menampilkan URL publik

In [ ]:
import subprocess
import time

def run_streamlit(filename, port=8501):
    subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
    subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True)
    ngrok.kill()
    time.sleep(3)

    proc = subprocess.Popen(
        [
            "streamlit", "run", filename,
            "--server.headless=true",
            "--server.port", str(port),
            "--server.enableCORS=false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    time.sleep(3)
    public_url = ngrok.connect(port)
    print(f"Streamlit berjalan: {public_url}")
    return proc

---

# Part 1: Komponen Dasar Streamlit

## Cara Kerja Streamlit

Streamlit bekerja dengan model yang unik: **setiap kali user berinteraksi dengan UI, seluruh skrip Python dijalankan ulang dari atas ke bawah.**

## Elemen-elemen UI Utama

### Teks & Judul
- `st.title()`, `st.header()`, `st.subheader()`, `st.write()`, `st.markdown()`

### Input dari User
- `st.text_input()`, `st.text_area()`, `st.button()`, `st.checkbox()`
- `st.selectbox()`, `st.slider()`, `st.file_uploader()`

### Layout
- `st.sidebar`, `st.columns()`, `st.expander()`

### Notifikasi
- `st.success()`, `st.info()`, `st.warning()`, `st.error()`

### Data & Chart
- `st.dataframe()`, `st.line_chart()`, `st.bar_chart()`, `st.area_chart()`

In [ ]:
%%writefile streamlit_app_basic.py
import streamlit as st
import pandas as pd
import numpy as np
import time

st.title("Streamlit Basic Tutorial")
st.write("""
## Selamat datang di Streamlit!
Ini adalah demo komponen-komponen dasar yang tersedia di Streamlit.
""")

# 1. Text Input
st.header("1. Text Input")
user_input = st.text_input("Masukkan namamu", "Ketik di sini...")
st.write(f"Halo, {user_input}!")

# 2. Button
st.header("2. Button")
if st.button("Klik aku!"):
    st.write("Tombol diklik!")

# 3. Checkbox
st.header("3. Checkbox")
show_content = st.checkbox("Tampilkan pesan rahasia")
if show_content:
    st.write("Kamu menemukan pesan rahasianya!")

# 4. Selectbox
st.header("4. Selectbox")
option = st.selectbox("Pilih warna favoritmu", ("Merah", "Biru", "Hijau", "Kuning"))
st.write(f"Kamu memilih: {option}")

# 5. Slider
st.header("5. Slider")
age = st.slider("Berapa umurmu?", 0, 100, 25)
st.write(f"Umurmu adalah {age} tahun")

# 6. Progress Bar
st.header("6. Progress Bar")
progress_bar = st.progress(0)
for i in range(100):
    time.sleep(0.01)
    progress_bar.progress(i + 1)
st.write("Selesai!")

# 7. Sidebar
st.header("7. Sidebar")
with st.sidebar:
    st.header("Panel Samping")
    if st.button("Tombol di Sidebar"):
        st.write("Tombol sidebar diklik!")

# 8. Columns
st.header("8. Columns")
col1, col2 = st.columns(2)
with col1:
    st.write("Ini kolom kiri")
    st.button("Tombol di kolom kiri")
with col2:
    st.write("Ini kolom kanan")
    st.button("Tombol di kolom kanan")

# 9. Status Messages
st.header("9. Status Messages")
st.success("Ini pesan sukses!")
st.info("Ini pesan informasi")
st.warning("Ini pesan peringatan")
st.error("Ini pesan error")

# 10. Charts
st.header("10. Charts")
chart_data = pd.DataFrame(np.random.randn(20, 3), columns=["Metrik A", "Metrik B", "Metrik C"])
st.line_chart(chart_data)

bar_data = pd.DataFrame(
    {"Apel": [10, 25, 18, 30], "Mangga": [15, 12, 22, 8]},
    index=["Jan", "Feb", "Mar", "Apr"]
)
st.bar_chart(bar_data)

# 11. Dataframe
st.header("11. Dataframe")
df = pd.DataFrame({
    "Nama":  ["Alice", "Bob", "Charlie", "Diana"],
    "Skor":  [88, 72, 95, 80],
    "Level": ["A", "B", "A+", "A"],
})
st.dataframe(df)
st.write(df.describe())

In [ ]:
proc = run_streamlit("streamlit_app_basic.py")

### Yang Perlu Diperhatikan

1. **Setiap interaksi menyebabkan skrip jalan ulang** — Progress bar akan berputar lagi setiap kali kamu klik sesuatu.
2. **Sidebar bisa disembunyikan** — klik panah kecil di pojok kiri atas app.
3. **Chart otomatis responsif** — coba resize jendela browser.

In [ ]:
try:
    proc.terminate()
    print("App dihentikan.")
except:
    pass

---

# Part 2: Chatbot dengan Gemini

## Konsep Kunci: st.session_state

Streamlit menjalankan ulang seluruh skrip setiap kali ada interaksi.
Artinya semua variabel Python akan di-reset ke nilai awal setiap kali user kirim pesan.

**Masalah:** Kalau pesan-pesan chat disimpan di variabel biasa, semua pesan akan hilang!

**Solusi:** `st.session_state` adalah dictionary khusus yang **nilainya tetap tersimpan** meskipun skrip dijalankan ulang.

```python
# Variabel biasa — akan HILANG setiap rerun
messages = []

# st.session_state — akan BERTAHAN setiap rerun
st.session_state.messages = []
```

## Komponen Chat Streamlit

```python
with st.chat_message("user"):
    st.markdown("pesan dari user")

with st.chat_message("assistant"):
    st.markdown("jawaban dari AI")

prompt = st.chat_input("Ketik pesanmu...")
```

## Alur Logic Chatbot

```
Skrip dijalankan (setiap interaksi)
         ↓
Cek API key di sidebar
         ↓
Inisialisasi client & chat session (hanya kalau belum ada di session_state)
         ↓
Tampilkan semua pesan dari st.session_state.messages
         ↓
Tunggu input dari st.chat_input()
         ↓ (user kirim pesan)
Tambah pesan user → Kirim ke Gemini → Tampilkan respons → Simpan ke messages
```

## Cara Mendapatkan Google AI API Key

1. Buka [https://aistudio.google.com](https://aistudio.google.com)
2. Klik **Get API Key** → **Create API Key**
3. Paste langsung ke kotak "Google AI API Key" di sidebar app

In [ ]:
%%writefile streamlit_chat_app.py
import streamlit as st
from google import genai

# ── 1. Konfigurasi Halaman ────────────────────────────────────────────────────
st.title("Gemini Chatbot")
st.caption("Chatbot sederhana menggunakan Google Gemini Flash")

# ── 2. Sidebar ────────────────────────────────────────────────────────────────
with st.sidebar:
    st.subheader("Pengaturan")
    google_api_key = st.text_input("Google AI API Key", type="password")
    reset_button = st.button("Reset Percakapan", help="Hapus semua pesan dan mulai dari awal")

# ── 3. Validasi API Key ───────────────────────────────────────────────────────
if not google_api_key:
    st.info("Masukkan Google AI API Key di sidebar untuk mulai chat.", icon="🗝️")
    st.stop()

# ── 4. Inisialisasi Gemini Client ─────────────────────────────────────────────
if ("genai_client" not in st.session_state) or (
    getattr(st.session_state, "_last_key", None) != google_api_key
):
    try:
        st.session_state.genai_client = genai.Client(api_key=google_api_key)
        st.session_state._last_key = google_api_key
        st.session_state.pop("chat", None)
        st.session_state.pop("messages", None)
    except Exception as e:
        st.error(f"API Key tidak valid: {e}")
        st.stop()

# ── 5. Inisialisasi Chat Session & Riwayat Pesan ─────────────────────────────
if "chat" not in st.session_state:
    st.session_state.chat = st.session_state.genai_client.chats.create(
        model="gemini-2.5-flash"
    )

if "messages" not in st.session_state:
    st.session_state.messages = []

# ── 6. Tombol Reset ───────────────────────────────────────────────────────────
if reset_button:
    st.session_state.pop("chat", None)
    st.session_state.pop("messages", None)
    st.rerun()

# ── 7. Tampilkan Riwayat Percakapan ──────────────────────────────────────────
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# ── 8. Input & Respons ────────────────────────────────────────────────────────
prompt = st.chat_input("Ketik pesanmu di sini...")

if prompt:
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    try:
        response = st.session_state.chat.send_message(prompt)
        answer = response.text if hasattr(response, "text") else str(response)
    except Exception as e:
        answer = f"Terjadi error: {e}"

    with st.chat_message("assistant"):
        st.markdown(answer)

    st.session_state.messages.append({"role": "assistant", "content": answer})

In [ ]:
proc = run_streamlit("streamlit_chat_app.py")

### Panduan Mencoba App

1. Buka URL yang muncul di atas
2. Di sidebar, masukkan Google AI API Key kamu
3. Mulai chat di kotak input bagian bawah
4. Coba tanya beberapa pertanyaan — perhatikan bahwa model **mengingat konteks** percakapan sebelumnya
5. Klik **Reset Percakapan** di sidebar — konteks akan hilang

### Yang Bisa Dikustomisasi

- **Ganti model** — ubah `"gemini-2.5-flash"` ke model lain
- **Tambah system prompt**:
  ```python
  st.session_state.chat = st.session_state.genai_client.chats.create(
      model="gemini-2.5-flash",
      config={"system_instruction": "Kamu adalah asisten yang selalu menjawab dalam Bahasa Indonesia"}
  )
  ```
- **Tambah parameter** di sidebar — misalnya slider untuk temperature

## Menghentikan App

In [ ]:
try:
    proc.terminate()
    print("Streamlit dihentikan.")
except:
    print("Tidak ada proses yang berjalan.")

ngrok.kill()
print("Tunnel ngrok ditutup.")

## Ringkasan

| Konsep | Penjelasan |
|---|---|
| `%%writefile` | Magic command Colab untuk menulis konten cell ke file di disk |
| `subprocess.Popen` | Menjalankan proses (Streamlit) di background tanpa memblokir notebook |
| `ngrok.connect()` | Membuat URL publik yang mengarah ke port lokal |
| `st.session_state` | Dictionary yang nilainya bertahan meskipun skrip dijalankan ulang |
| `st.chat_message()` | Membuat bubble chat dengan role user/assistant |
| `st.chat_input()` | Kotak input yang muncul di bagian bawah halaman |
| `st.stop()` | Menghentikan eksekusi skrip di titik tersebut |
| `st.rerun()` | Memaksa Streamlit me-refresh halaman dari awal |

## Referensi

- [Dokumentasi Streamlit](https://docs.streamlit.io)
- [Streamlit Chat Elements](https://docs.streamlit.io/develop/api-reference/chat)
- [Session State](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.session_state)